# 🌊 VLM Flood Detection Benchmark: Qwen 2.5 VL vs. LLaVA 1.5
### Hallucination Rate, Faithfulness, and Citation Evaluation on Satellite Imagery

This notebook benchmarks two open-weights Vision-Language Models on the **Sen1Floods11** optical dataset ($128 \times 128$ resolution):
1. **Model 1: Qwen2.5-VL-7B-Instruct** (Native bounding boxes & spatial grounding)
2. **Model 2: LLaVA-1.5-7B-hf** (Standard academic benchmark for VLM hallucination)

### ⚡ Hardware Requirements
Run on Google Colab with a GPU enabled:
**Runtime -> Change runtime type -> T4 GPU** (or L4 / A100 if available).
Both 7B models are loaded with **4-bit quantization (`bitsandbytes`)** so they each consume only ~5 GB VRAM and run comfortably on the free T4 GPU.

## 📦 Step 1: Install Dependencies
Install Hugging Face libraries, `bitsandbytes` (for 4-bit quantization), and `qwen-vl-utils`.

In [ ]:
# 1. Install GPU-compatible dependencies cleanly
!pip install -q -U "pillow==11.3.0"
!pip install -q -U torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q -U transformers accelerate bitsandbytes qwen-vl-utils

import sys
for key in list(sys.modules.keys()):
    if key.startswith("PIL") or key.startswith("pillow"):
        sys.modules.pop(key, None)

import torch
print("PyTorch Version:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 12.5 MB/s eta 0:00:00
PyTorch Version: 2.11.0+cu130
GPU Available: True
Device: Tesla T4
VRAM: 15.64 GB


In [ ]:
import os
import sys

# Programmatically restart Colab session to refresh compiled C-extensions (like _imaging)
print("🔄 Restarting the Colab runtime kernel to refresh PIL extensions... Please wait a few seconds, then execute Step 8 (Qwen VL evaluation) again!")
os.kill(os.getpid(), 9)

## 💾 Step 2: Google Drive Mounting & Persistent Model Caching

### Why Persistent Caching Matters
By default, Colab downloads 15 GB model weights into ephemeral storage, which is wiped when the session disconnects.
By setting `HF_HOME` to Google Drive, **the model weights are downloaded once into Drive and load in seconds on every future run**.

In [ ]:
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Configure paths on Google Drive
DRIVE_ROOT = "/content/drive/MyDrive/vlm_flood_research"
DRIVE_CACHE = f"{DRIVE_ROOT}/hf_cache"
DRIVE_DATA = f"{DRIVE_ROOT}/data"
DRIVE_RESULTS = f"{DRIVE_ROOT}/results"

os.environ["HF_HOME"] = DRIVE_CACHE
os.makedirs(DRIVE_CACHE, exist_ok=True)
os.makedirs(DRIVE_DATA, exist_ok=True)
os.makedirs(DRIVE_RESULTS, exist_ok=True)

print(f"✅ Google Drive mounted successfully!")
print(f"📁 Persistent Hugging Face cache: {DRIVE_CACHE}")
print(f"📁 Results directory: {DRIVE_RESULTS}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted successfully!
📁 Persistent Hugging Face cache: /content/drive/MyDrive/vlm_flood_research/hf_cache
📁 Results directory: /content/drive/MyDrive/vlm_flood_research/results


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 🗂️ Step 3: Dataset Setup on Fast Local SSD
If you uploaded `sen1floods_128.zip` to `MyDrive/vlm_flood_research/data/sen1floods_128.zip`, we extract it to Colab's fast local SSD (`/content/data/`) for lightning-fast image I/O.

In [ ]:
import zipfile
import json
from pathlib import Path

zip_path = f"{DRIVE_DATA}/sen1floods_128.zip"
local_data_dir = Path("/content/data")

if os.path.exists(zip_path):
    print(f"Found dataset archive at {zip_path}. Extracting to {local_data_dir}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(local_data_dir)
    print("Extraction complete!")
else:
    print(f"Archive not found at {zip_path}.")
    print("Checking direct folder in Drive...")
    if os.path.exists(f"{DRIVE_DATA}/processed_128"):
        local_data_dir = Path(DRIVE_DATA)
        print(f"Using dataset directly from Drive: {local_data_dir}")
    else:
        print("⚠️ Please upload 'sen1floods_128.zip' to Drive or check the folder path.")

# Load metadata
meta_path = local_data_dir / "processed_128" / "metadata.json"
if meta_path.exists():
    with open(meta_path, 'r') as f:
        metadata = json.load(f)
    print(f"Loaded metadata for {len(metadata)} 128x128 image tiles.")
else:
    metadata = {}
    print(f"Metadata not found yet at {meta_path}.")

Found dataset archive at /content/drive/MyDrive/vlm_flood_research/data/sen1floods_128.zip. Extracting to /content/data...
Extraction complete!
Loaded metadata for 6816 128x128 image tiles.


In [ ]:
# @title
import os
from pathlib import Path

# Let's inspect the extracted images to see if they are 0-byte files or corrupted
local_img_dir = Path("/content/data/processed_128")

if local_img_dir.exists():
    files = sorted(os.listdir(local_img_dir))
    print(f"Total files in {local_img_dir}: {len(files)}")

    # Check first 10 files and their sizes
    print("\nFirst 10 files and their sizes:")
    for f in files[:10]:
        fpath = local_img_dir / f
        sz = os.path.getsize(fpath)
        print(f"  - {f}: {sz} bytes (Size: {sz / 1024:.2f} KB)")

    # If the file sizes are 0 or extremely small, the extraction or zip archive on Drive is corrupted.
    zip_path = f"{DRIVE_DATA}/sen1floods_128.zip"
    if os.path.exists(zip_path):
        print(f"\nZip archive size on Drive: {os.path.getsize(zip_path) / (1024*1024):.2f} MB")
else:
    print(f"Directory {local_img_dir} does not exist. Please run Step 3 again.")

Total files in /content/data/processed_128: 6817

First 10 files and their sizes:
  - Bolivia_103757_r0_c2.png: 2943 bytes (Size: 2.87 KB)
  - Bolivia_103757_r0_c3.png: 26001 bytes (Size: 25.39 KB)
  - Bolivia_103757_r1_c2.png: 7834 bytes (Size: 7.65 KB)
  - Bolivia_103757_r1_c3.png: 24540 bytes (Size: 23.96 KB)
  - Bolivia_103757_r2_c2.png: 14646 bytes (Size: 14.30 KB)
  - Bolivia_103757_r2_c3.png: 23544 bytes (Size: 22.99 KB)
  - Bolivia_103757_r3_c2.png: 19157 bytes (Size: 18.71 KB)
  - Bolivia_103757_r3_c3.png: 21969 bytes (Size: 21.45 KB)
  - Bolivia_129334_r0_c0.png: 26499 bytes (Size: 25.88 KB)
  - Bolivia_129334_r0_c1.png: 24061 bytes (Size: 23.50 KB)

Zip archive size on Drive: 354.04 MB


## ⚙️ Step 4: Shared 4-Bit Quantization Configuration
Uses `bitsandbytes` NormalFloat4 (NF4) with FP16 compute dtype to compress both 7B parameter models from ~15 GB VRAM down to **~5 GB VRAM** with zero noticeable reasoning degradation.

In [ ]:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)
print("BitsAndBytes 4-bit configuration initialized.")

BitsAndBytes 4-bit configuration initialized.


In [ ]:
import shutil
import os

# Define the exact model cache folder to purge
qwen_cache_path = os.path.join(
    os.environ.get("HF_HOME", "/content/drive/MyDrive/vlm_flood_research/hf_cache"),
    "hub/models--Qwen--Qwen2.5-VL-7B-Instruct"
)

if os.path.exists(qwen_cache_path):
    print(f"🧹 Removing corrupted Qwen cache directory to force a clean re-download: {qwen_cache_path}")
    try:
        shutil.rmtree(qwen_cache_path)
        print("✅ Corrupted cache folder deleted successfully! Please run Step 8 (Qwen VL Cell) again to re-download clean weights.")
    except Exception as e:
        print(f"❌ Error while deleting folder: {e}")
else:
    print(f"Qwen cache folder not found at {qwen_cache_path}. Make sure HF_HOME matches your cache location.")

🧹 Removing corrupted Qwen cache directory to force a clean re-download: /content/drive/MyDrive/vlm_flood_research/hf_cache/hub/models--Qwen--Qwen2.5-VL-7B-Instruct
✅ Corrupted cache folder deleted successfully! Please run Step 8 (Qwen VL Cell) again to re-download clean weights.


## 🤖 Step 5: Model 1 — Qwen2.5-VL-7B-Instruct
Loads Qwen2.5-VL and defines its inference function supporting dynamic image inputs.

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
from PIL import Image

def load_qwen_model():
    print("Loading Qwen2.5-VL-7B-Instruct (4-bit NF4)... Weights cached to Drive.")
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        "Qwen/Qwen2.5-VL-7B-Instruct",
        device_map="auto",
        quantization_config=bnb_config
    )
    processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")
    print("Qwen 2.5 VL loaded successfully!")
    return model, processor

def infer_qwen(model, processor, image_path, prompt):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": f"file://{os.path.abspath(image_path)}"},
                {"type": "text", "text": prompt},
            ],
        }
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=128)
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    return output[0].strip()

## 🤖 Step 6: Model 2 — InternVL2.5-8B
Loads InternVL2.5-8B and defines its inference function.

In [ ]:
from transformers import LlavaForConditionalGeneration, AutoProcessor
from PIL import Image
import torch

def load_llava_model():
    print("Loading LLaVA-1.5-7B-hf (4-bit NF4)... Weights cached to Drive.")
    model = LlavaForConditionalGeneration.from_pretrained(
        "llava-hf/llava-1.5-7b-hf",
        device_map="auto",
        quantization_config=bnb_config
    )
    processor = AutoProcessor.from_pretrained(
        "llava-hf/llava-1.5-7b-hf"
    )
    print("LLaVA 1.5 loaded successfully!")
    return model, processor

def infer_llava(model, processor, image_path, prompt):
    raw_image = Image.open(image_path).convert("RGB")
    formatted_prompt = f"USER: <image>\n{prompt}\nASSISTANT:"
    inputs = processor(text=formatted_prompt, images=raw_image, return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=128)
    generated_ids_trimmed = generated_ids[0][inputs.input_ids.shape[1]:]
    output = processor.decode(generated_ids_trimmed, skip_special_tokens=True)
    return output.strip()

## 🚀 Step 7: Turnkey Batch Inference Engine (Streaming Checkpoints to Drive)

This engine executes the **Two-Stage Evaluation**:
* **Stage 1 (Normal Images):** `processed_128/` (Water pixels visible)
* **Stage 2 (Counterfactual Masked Images):** `processed_128_covered/` (Water pixels replaced with dry land)

### 🛡️ Crash & Disconnect Protection
Every prediction is saved line-by-line as a JSON Line (`.jsonl`) file directly in your Google Drive (`/content/drive/MyDrive/vlm_flood_research/results/`). If Colab disconnects, **re-running the cell automatically resumes without repeating any image**!

In [ ]:
import time
from PIL import Image
import json
import os

def run_batch_evaluation(model_name, model, processor, infer_fn, stage="stage1", max_samples=None):
    """
    Runs batch inference for a given model and stage.
    stage: 'stage1' (Normal images) or 'stage2' (Masked images)
    """
    folder_name = "processed_128" if stage == "stage1" else "processed_128_covered"
    images_dir = local_data_dir / folder_name

    out_file = f" {DRIVE_RESULTS}/{model_name}_{stage}_results.jsonl".strip()
    print(f"\n{'='*60}")
    print(f"Running {model_name} on {stage.upper()} ({folder_name})")
    print(f"Output Checkpoint: {out_file}")
    print(f"{'='*60}")

    # 1. Read existing checkpoints to resume seamlessly
    completed_ids = set()
    if os.path.exists(out_file):
        with open(out_file, "r") as f:
            for line in f:
                if line.strip():
                    try:
                        completed_ids.add(json.loads(line)["tile_id"])
                    except Exception:
                        pass
        print(f"Found {len(completed_ids)} already completed chips. Resuming...")

    # 2. Select items to process
    tile_ids = list(metadata.keys())
    if max_samples:
        tile_ids = tile_ids[:max_samples]

    remaining_ids = [tid for tid in tile_ids if tid not in completed_ids]
    print(f"Total items: {len(tile_ids)} | Remaining to run: {len(remaining_ids)}")

    # 3. Batch loop with streaming append
    start_time = time.time()
    skipped_count = 0
    with open(out_file, "a", buffering=1) as out_f:
        for idx, tid in enumerate(remaining_ids):
            item = metadata[tid]
            img_path = images_dir / f"{tid}.png"

            if not img_path.exists():
                continue

            # Validate image integrity before running inference
            try:
                with Image.open(img_path) as test_img:
                    test_img.verify()
            except Exception as e:
                print(f"⚠️ Skipping corrupted image {img_path.name}: {e}")
                skipped_count += 1
                continue

            pope_prompt = item["prompts"]["pope"]
            json_prompt = item["prompts"]["structured_json"]

            # Execute inference
            try:
                t0 = time.time()
                pred_pope = infer_fn(model, processor, str(img_path), pope_prompt)
                pred_json = infer_fn(model, processor, str(img_path), json_prompt)
                latency = time.time() - t0

                record = {
                    "tile_id": tid,
                    "parent_chip_id": item["parent_chip_id"],
                    "stage": stage,
                    "model": model_name,
                    "image_file": f"{folder_name}/{tid}.png",
                    "ground_truth": {
                        "water_fraction": item["stats"]["water_fraction"],
                        "dominant_quadrant": item["stats"]["dominant_quadrant"],
                        "flood_present_gt": item["stats"]["flood_present"],
                        "flood_tier": item["stats"]["flood_tier"],
                        "is_mostly_cloud": item["stats"]["is_mostly_cloud"]
                    },
                    "predictions": {
                        "pope_raw": pred_pope,
                        "json_raw": pred_json
                    },
                    "latency_sec": round(latency, 3)
                }

                out_f.write(json.dumps(record) + "\n")
            except Exception as eval_error:
                print(f"❌ Error predicting for {tid}: {eval_error}")
                continue

            if (idx + 1) % 25 == 0 or (idx + 1) == len(remaining_ids):
                elapsed = time.time() - start_time
                speed = (idx + 1 - skipped_count) / max(elapsed, 1e-5)
                print(f"[{idx + 1}/{len(remaining_ids)}] Completed {tid} | Speed: {speed:.1f} tiles/sec | POPE: {pred_pope[:20]}")

    print(f"\nFinished {stage} for {model_name}. Skipped {skipped_count} corrupted images.")
    print(f"Checkpoint saved to Drive: {out_file}")

## 🎯 Step 8: Execute Qwen 2.5 VL (Stage 1 & Stage 2)
Run inference across the normal and masked datasets. You can specify `max_samples=200` for a fast initial test, or set `max_samples=None` for the full dataset.

In [ ]:
# 1. Load Qwen 2.5 VL
qwen_model, qwen_processor = load_qwen_model()

# 2. Run Stage 1 (Normal images)
run_batch_evaluation(
    model_name="qwen2_5_vl_7b",
    model=qwen_model,
    processor=qwen_processor,
    infer_fn=infer_qwen,
    stage="stage1",
    max_samples=1500  # Set to None for full run
)

# 3. Run Stage 2 (Masked Counterfactual images)
run_batch_evaluation(
    model_name="qwen2_5_vl_7b",
    model=qwen_model,
    processor=qwen_processor,
    infer_fn=infer_qwen,
    stage="stage2",
    max_samples=1500  # Set to None for full run
)

# Clean up VRAM before loading Model 2
import gc
del qwen_model
del qwen_processor
gc.collect()
torch.cuda.empty_cache()
print("Cleaned up Qwen from VRAM. Ready for LLaVA!")

Loading Qwen2.5-VL-7B-Instruct (4-bit NF4)... Weights cached to Drive.


config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/57.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Qwen 2.5 VL loaded successfully!

Running qwen2_5_vl_7b on STAGE1 (processed_128)
Output Checkpoint: /content/drive/MyDrive/vlm_flood_research/results/qwen2_5_vl_7b_stage1_results.jsonl
Found 0 already completed chips. Resuming...
Total items: 1500 | Remaining to run: 1500


/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (3420) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


[25/1500] Completed Bolivia_195474_r0_c0 | Speed: 0.3 tiles/sec | POPE: Yes.
[50/1500] Completed Bolivia_23014_r2_c1 | Speed: 0.3 tiles/sec | POPE: No.
[75/1500] Completed Bolivia_242570_r0_c3 | Speed: 0.3 tiles/sec | POPE: No.
[100/1500] Completed Bolivia_290290_r3_c0 | Speed: 0.3 tiles/sec | POPE: Yes.
[125/1500] Completed Bolivia_312675_r1_c1 | Speed: 0.3 tiles/sec | POPE: Yes.
[150/1500] Completed Bolivia_314919_r3_c2 | Speed: 0.3 tiles/sec | POPE: Yes.
[175/1500] Completed Bolivia_379434_r1_c3 | Speed: 0.3 tiles/sec | POPE: No.
[200/1500] Completed Bolivia_60373_r0_c1 | Speed: 0.4 tiles/sec | POPE: Yes.
[225/1500] Completed Bolivia_76104_r2_c2 | Speed: 0.4 tiles/sec | POPE: Yes.
[250/1500] Completed Ghana_1033830_r0_c3 | Speed: 0.4 tiles/sec | POPE: No.
[275/1500] Completed Ghana_1078550_r3_c0 | Speed: 0.4 tiles/sec | POPE: No.
[300/1500] Completed Ghana_11745_r1_c2 | Speed: 0.4 tiles/sec | POPE: No.
[325/1500] Completed Ghana_124834_r3_c3 | Speed: 0.4 tiles/sec | POPE: Yes.
[350/

## 🎯 Step 9: Execute InternVL 2.5 (Stage 1 & Stage 2)
Loads InternVL 2.5 and executes the same evaluation across both normal and masked sets.

In [ ]:
import gc
import torch
import os
import json
import time
from pathlib import Path
from PIL import Image
from google.colab import drive

# 1. Install specific fully compatible versions of the ecosystem packages
!pip install -q -U "huggingface-hub>=0.25.0" "transformers==4.46.3" "accelerate>=0.26.0" "bitsandbytes>=0.42.0"

# 2. Safely patch bitsandbytes multi-backend validation to bypass the frozenset bug
import transformers
try:
    # Inject dummy function directly into transformers utils if queried
    transformers.utils.is_bitsandbytes_multi_backend_available = lambda: False
except AttributeError:
    pass

from transformers import LlavaForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

# 3. Force garbage collection and VRAM flush
gc.collect()
torch.cuda.empty_cache()

# 4. Ensure Drive is mounted and metadata variables are loaded dynamically
try:
    drive.mount('/content/drive')
except Exception:
    pass

DRIVE_ROOT = "/content/drive/MyDrive/vlm_flood_research"
DRIVE_RESULTS = f"{DRIVE_ROOT}/results"
local_data_dir = Path("/content/data")

meta_path = local_data_dir / "processed_128" / "metadata.json"
if meta_path.exists():
    with open(meta_path, 'r') as f:
        metadata = json.load(f)
else:
    meta_path = Path(DRIVE_ROOT) / "data" / "processed_128" / "metadata.json"
    if meta_path.exists():
        with open(meta_path, 'r') as f:
            metadata = json.load(f)
    else:
        raise FileNotFoundError("Metadata JSON not found! Please run Step 3 to extract the dataset.")

# 5. Initialize local bnb_config safely
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# 6. Define LLaVA Helpers locally
def load_llava_model():
    print("Loading LLaVA-1.5-7B-hf (4-bit NF4)... Weights cached to Drive.")
    model = LlavaForConditionalGeneration.from_pretrained(
        "llava-hf/llava-1.5-7b-hf",
        device_map="auto",
        quantization_config=bnb_config
    )
    processor = AutoProcessor.from_pretrained("llava-hf/llava-1.5-7b-hf")
    processor.patch_size = 14
    processor.vision_feature_select_strategy = "default"
    print("LLaVA 1.5 loaded successfully!")
    return model, processor

def infer_llava(model, processor, image_path, prompt):
    raw_image = Image.open(image_path).convert("RGB")
    formatted_prompt = f"USER: <image>\n{prompt}\nASSISTANT:"
    inputs = processor(text=formatted_prompt, images=raw_image, return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=128)
    generated_ids_trimmed = generated_ids[0][inputs.input_ids.shape[1]:]
    output = processor.decode(generated_ids_trimmed, skip_special_tokens=True)
    return output.strip()

# 7. Verbose Batch evaluation function
def run_batch_evaluation_verbose(model_name, model, processor, infer_fn, stage="stage1", max_samples=None):
    folder_name = "processed_128" if stage == "stage1" else "processed_128_covered"
    images_dir = local_data_dir / folder_name

    if not images_dir.exists():
        print(f"⚠️ Warning: Directory {images_dir} does not exist!")
    else:
        num_local_images = len(os.listdir(images_dir))
        print(f"Verified local folder: {images_dir} containing {num_local_images} images.")

    out_file = f"{DRIVE_RESULTS}/{model_name}_{stage}_results.jsonl".strip()
    print(f"\n{'='*60}")
    print(f"Running {model_name} on {stage.upper()} ({folder_name})")
    print(f"Output Checkpoint: {out_file}")
    print(f"{'='*60}")

    completed_ids = set()
    if os.path.exists(out_file):
        with open(out_file, "r") as f:
            for line in f:
                if line.strip():
                    try:
                        completed_ids.add(json.loads(line)["tile_id"])
                    except Exception:
                        pass
        print(f"Found {len(completed_ids)} already completed chips. Resuming...")

    tile_ids = list(metadata.keys())
    if max_samples:
        tile_ids = tile_ids[:max_samples]

    remaining_ids = [tid for tid in tile_ids if tid not in completed_ids]
    print(f"Total items: {len(tile_ids)} | Remaining to run: {len(remaining_ids)}")

    skipped_count = 0
    with open(out_file, "a", buffering=1) as out_f:
        for idx, tid in enumerate(remaining_ids):
            item = metadata[tid]
            img_path = images_dir / f"{tid}.png"

            if not img_path.exists():
                skipped_count += 1
                continue

            try:
                with Image.open(img_path) as test_img:
                    test_img.verify()
            except Exception as e:
                print(f"⚠️ Skipping corrupted image {img_path.name}: {e}")
                skipped_count += 1
                continue

            pope_prompt = item["prompts"]["pope"]
            json_prompt = item["prompts"]["structured_json"]

            try:
                t0 = time.time()
                pred_pope = infer_fn(model, processor, str(img_path), pope_prompt)
                pred_json = infer_fn(model, processor, str(img_path), json_prompt)
                latency = time.time() - t0

                record = {
                    "tile_id": tid,
                    "parent_chip_id": item["parent_chip_id"],
                    "stage": stage,
                    "model": model_name,
                    "image_file": f"{folder_name}/{tid}.png",
                    "ground_truth": {
                        "water_fraction": item["stats"]["water_fraction"],
                        "dominant_quadrant": item["stats"]["dominant_quadrant"],
                        "flood_present_gt": item["stats"]["flood_present"],
                        "flood_tier": item["stats"]["flood_tier"],
                        "is_mostly_cloud": item["stats"]["is_mostly_cloud"]
                    },
                    "predictions": {
                        "pope_raw": pred_pope,
                        "json_raw": pred_json
                    },
                    "latency_sec": round(latency, 3)
                }

                out_f.write(json.dumps(record) + "\n")
                print(f"[Processed {idx+1}/{len(remaining_ids)}] {tid} | POPE Answer: {pred_pope[:30]} | Latency: {latency:.2f}s")
            except Exception as eval_error:
                print(f"❌ Error predicting for {tid}: {eval_error}")
                continue

    print(f"\nFinished {stage} for {model_name}. Skipped/Missing {skipped_count} images.")

# 8. Load LLaVA 1.5 model safely
llava_model, llava_processor = load_llava_model()

# 9. Run Stage 1 (Normal images)
run_batch_evaluation_verbose(
    model_name="llava_1_5_7b",
    model=llava_model,
    processor=llava_processor,
    infer_fn=infer_llava,
    stage="stage1",
    max_samples=1500
)

# 10. Run Stage 2 (Masked Counterfactual images)
run_batch_evaluation_verbose(
    model_name="llava_1_5_7b",
    model=llava_model,
    processor=llava_processor,
    infer_fn=infer_llava,
    stage="stage2",
    max_samples=1500
)

print("🎉 All inference runs complete!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading LLaVA-1.5-7B-hf (4-bit NF4)... Weights cached to Drive.


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

Some kwargs in processor config are unused and will not have any effect: num_additional_image_tokens. 


LLaVA 1.5 loaded successfully!
Verified local folder: /content/data/processed_128 containing 6817 images.

Running llava_1_5_7b on STAGE1 (processed_128)
Output Checkpoint: /content/drive/MyDrive/vlm_flood_research/results/llava_1_5_7b_stage1_results.jsonl
Total items: 1500 | Remaining to run: 1500


Expanding inputs for image tokens in LLaVa should be done in processing. Please add `patch_size` and `vision_feature_select_strategy` to the model's processing config or set directly with `processor.patch_size = {{patch_size}}` and processor.vision_feature_select_strategy = {{vision_feature_select_strategy}}`. Using processors without these attributes in the config is deprecated and will throw an error in v4.47.


[Processed 1/1500] Bolivia_103757_r0_c2 | POPE Answer: No | Latency: 5.51s
[Processed 2/1500] Bolivia_103757_r0_c3 | POPE Answer: Yes | Latency: 2.82s
[Processed 3/1500] Bolivia_103757_r1_c2 | POPE Answer: Yes | Latency: 2.77s
[Processed 4/1500] Bolivia_103757_r1_c3 | POPE Answer: Yes | Latency: 2.77s
[Processed 5/1500] Bolivia_103757_r2_c2 | POPE Answer: Yes | Latency: 2.87s
[Processed 6/1500] Bolivia_103757_r2_c3 | POPE Answer: Yes | Latency: 2.83s
[Processed 7/1500] Bolivia_103757_r3_c2 | POPE Answer: Yes | Latency: 2.83s
[Processed 8/1500] Bolivia_103757_r3_c3 | POPE Answer: No | Latency: 2.89s
[Processed 9/1500] Bolivia_129334_r0_c0 | POPE Answer: Yes | Latency: 2.82s
[Processed 10/1500] Bolivia_129334_r0_c1 | POPE Answer: Yes | Latency: 2.81s
[Processed 11/1500] Bolivia_129334_r0_c2 | POPE Answer: No | Latency: 2.79s
[Processed 12/1500] Bolivia_129334_r0_c3 | POPE Answer: No | Latency: 2.85s
[Processed 13/1500] Bolivia_129334_r1_c0 | POPE Answer: No | Latency: 2.79s
[Processed 14

In [ ]:
import os
print("🔄 Restarting the Colab runtime kernel to apply the clean library installations... Please wait 3 seconds, then run Step 9 (cell m_HErDt1HqaS) again!")
os.kill(os.getpid(), 9)

## 📊 Step 10: Quick Verification Dashboard
Inspects the completed JSONL logs to verify that predictions are ready for evaluation in Phase 5.

In [ ]:
print("=== Summary of Completed Inference Checkpoints in Google Drive ===")
for fname in os.listdir(DRIVE_RESULTS):
    if fname.endswith(".jsonl"):
        fpath = os.path.join(DRIVE_RESULTS, fname)
        with open(fpath, "r") as f:
            lines = [l for l in f if l.strip()]
        print(f"  - {fname:35s}: {len(lines):5d} predictions logged")

# Show a sample prediction from the first available log
log_files = [f for f in os.listdir(DRIVE_RESULTS) if f.endswith(".jsonl")]
if log_files:
    sample_log = os.path.join(DRIVE_RESULTS, log_files[0])
    with open(sample_log, "r") as f:
        first_line = json.loads(f.readline())
    print(f"\nSample entry from {log_files[0]}:")
    print(json.dumps(first_line, indent=2))

=== Summary of Completed Inference Checkpoints in Google Drive ===
  - llava_1_5_7b_stage1_results.jsonl  :  1500 predictions logged
  - llava_1_5_7b_stage2_results.jsonl  :  1500 predictions logged
  - qwen2_5_vl_7b_stage1_results.jsonl :  1500 predictions logged
  - qwen2_5_vl_7b_stage2_results.jsonl :  1500 predictions logged

Sample entry from llava_1_5_7b_stage1_results.jsonl:
{
  "tile_id": "Bolivia_103757_r0_c2",
  "parent_chip_id": "Bolivia_103757",
  "stage": "stage1",
  "model": "llava_1_5_7b",
  "image_file": "processed_128/Bolivia_103757_r0_c2.png",
  "ground_truth": {
    "water_fraction": 0.8537,
    "dominant_quadrant": "bottom-right",
    "flood_present_gt": true,
    "flood_tier": "cloud_obscured",
    "is_mostly_cloud": true
  },
  "predictions": {
    "pope_raw": "No",
    "json_raw": "{\n\"flood\\_present\": false,\n\"water\\_percentage\": 0,\n\"location\": \"none\"\n}"
  },
  "latency_sec": 5.514
}
